# CardioIA – Fase 2 | Parte 2: Classificador básico de texto (risco)

Objetivo: classificar relatos de pacientes como **alto risco** ou **baixo risco** usando **TF-IDF + modelo de Machine Learning (scikit-learn)**, simulando uma triagem clínica automatizada.

> ⚠️ Projeto **acadêmico**, com base de dados **simulada e pequena**. Não serve para uso clínico real.

## 1. Base de dados rotulada

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

dados = pd.read_csv("classificador_risco.csv", sep=";", encoding="utf-8-sig")
print("Total de frases:", len(dados))
print(dados["situacao"].value_counts().to_string())   # classes equilibradas?
display(dados)

Total de frases: 40
situacao
alto risco     20
baixo risco    20


                                                                         frase     situacao
0                                   sinto uma forte dor no peito e falta de ar   alto risco
1                                           estou com dor no peito e suor frio   alto risco
2                         meu coração está muito acelerado e estou com tontura   alto risco
3                                    sinto aperto no peito quando faço esforço   alto risco
4                             estou com muita falta de ar mesmo estando parado   alto risco
5                                        desmaiei depois de sentir palpitações   alto risco
6                          estou com dor forte no peito que começou de repente   alto risco
7                               sinto falta de ar e uma forte pressão no peito   alto risco
8                       estou com palpitações acompanhadas de fraqueza intensa   alto risco
9                             sinto dor no peito que está piorando rapidamente  

## 2. Separação em treino e teste

70% das frases para treinar e 30% para testar. O parâmetro `stratify` mantém a mesma proporção de alto/baixo risco nos dois conjuntos.

In [2]:
from sklearn.model_selection import train_test_split

X = dados["frase"]
y = dados["situacao"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
print("Frases para treinamento:", len(X_treino))
print("Frases para teste:", len(X_teste))

Frases para treinamento: 28
Frases para teste: 12


## 3. TF-IDF: transformando texto em números

O **TF-IDF** dá mais peso às palavras que aparecem muito em uma frase mas pouco no conjunto todo. O vocabulário é aprendido **somente com o treino** (para o teste não "vazar" informação).

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(ngram_range=(1, 2))      # palavras isoladas e pares de palavras ("dor no", "no peito")
X_treino_tfidf = tfidf.fit_transform(X_treino)
X_teste_tfidf = tfidf.transform(X_teste)

print("Tamanho do vocabulário:", len(tfidf.get_feature_names_out()))
print("Formato da matriz de treino:", X_treino_tfidf.shape)

Tamanho do vocabulário: 283
Formato da matriz de treino: (28, 283)


## 4. Treinamento: Regressão Logística e Árvore de Decisão

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

modelos = {
    "Regressão Logística": LogisticRegression(random_state=42),
    "Árvore de Decisão": DecisionTreeClassifier(random_state=42),
}
for nome, m in modelos.items():
    m.fit(X_treino_tfidf, y_treino)
    acc = accuracy_score(y_teste, m.predict(X_teste_tfidf))
    print(f"{nome}: acurácia no teste = {acc:.2%}")

modelo = modelos["Regressão Logística"]      # modelo principal
previsoes = modelo.predict(X_teste_tfidf)

Regressão Logística: acurácia no teste = 100.00%
Árvore de Decisão: acurácia no teste = 75.00%


## 5. Avaliação no conjunto de teste

In [5]:
resultado = pd.DataFrame({"Frase": X_teste, "Resposta correta": y_teste, "Previsão da IA": previsoes})
resultado["Acertou?"] = np.where(resultado["Resposta correta"] == resultado["Previsão da IA"], "sim", "NÃO")
display(resultado)

                                                                         Frase Resposta correta Previsão da IA Acertou?
14                        meu coração dispara sozinho e sinto o peito apertado       alto risco     alto risco      sim
7                               sinto falta de ar e uma forte pressão no peito       alto risco     alto risco      sim
20                                     tive um leve cansaço depois de caminhar      baixo risco    baixo risco      sim
4                             estou com muita falta de ar mesmo estando parado       alto risco     alto risco      sim
37                         estou com uma dor de cabeça leve por causa do calor      baixo risco    baixo risco      sim
28                   estou me sentindo cansado depois de uma noite mal dormida      baixo risco    baixo risco      sim
38  senti um ligeiro aperto no peito durante uma discussão e passou em minutos      baixo risco    baixo risco      sim
6                          estou com dor

In [6]:
from sklearn.metrics import classification_report, confusion_matrix

print("Acurácia:", f"{accuracy_score(y_teste, previsoes):.2%}\n")
print(classification_report(y_teste, previsoes, zero_division=0))

matriz = confusion_matrix(y_teste, previsoes, labels=["baixo risco", "alto risco"])
display(pd.DataFrame(matriz,
                     index=["Real: baixo risco", "Real: alto risco"],
                     columns=["Previsto: baixo risco", "Previsto: alto risco"]))

Acurácia: 100.00%

              precision    recall  f1-score   support

  alto risco       1.00      1.00      1.00         6
 baixo risco       1.00      1.00      1.00         6

    accuracy                           1.00        12
   macro avg       1.00      1.00      1.00        12
weighted avg       1.00      1.00      1.00        12



                   Previsto: baixo risco  Previsto: alto risco
Real: baixo risco                      6                     0
Real: alto risco                       0                     6

### Validação cruzada

Com apenas 12 frases de teste, o resultado de uma única divisão pode ser sorte ou azar. A **validação cruzada (5 partes)** treina e testa 5 vezes em divisões diferentes e dá uma estimativa mais confiável.

In [7]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

pipeline = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression(random_state=42))
validacao = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
notas = cross_val_score(pipeline, X, y, cv=validacao)

print("Acurácia em cada divisão:", np.round(notas, 2))
print(f"Média: {notas.mean():.2%}  (desvio padrão: {notas.std():.2%})")

Acurácia em cada divisão: [0.88 0.88 1.   0.88 1.  ]
Média: 92.50%  (desvio padrão: 6.12%)


## 6. O que o modelo aprendeu? (padrões e possíveis distorções)

Vamos ver quais palavras mais puxam a previsão para cada classe.

In [8]:
termos = tfidf.get_feature_names_out()
pesos = modelo.coef_[0]
# classes_ está em ordem alfabética: ['alto risco', 'baixo risco'] -> peso positivo = baixo risco
ordem = np.argsort(pesos)
print("Termos que mais indicam ALTO risco :", [termos[i] for i in ordem[:10]])
print("Termos que mais indicam BAIXO risco:", [termos[i] for i in ordem[::-1][:10]])

Termos que mais indicam ALTO risco : ['dor no', 'peito', 'no peito', 'está', 'palpitações', 'dor', 'suor', 'intensa', 'respirar', 'de ar']
Termos que mais indicam BAIXO risco: ['leve', 'um', 'depois', 'um pouco', 'pouco', 'com um', 'depois de', 'senti uma', 'senti', 'cansaço']


## 7. Testando com frases novas (que o modelo nunca viu)

In [9]:
novas_frases = [
    "estou com uma forte dor no peito e muita falta de ar",
    "estou apenas um pouco cansado depois de caminhar",
    "senti uma leve tontura mas já estou melhor",
    "meu coração está muito acelerado e estou me sentindo fraco",
    "meu braço esquerdo está dormente e sinto náusea",
    "tive uma leve dor no peito e falta de ar",
    "não sinto dor no peito",
    "estou com muita dor de cabeça",
]
probs = modelo.predict_proba(tfidf.transform(novas_frases))
for frase, p in zip(novas_frases, probs):
    classe = modelo.classes_[p.argmax()]
    print(f"{frase!r}\n   -> {classe} (confiança {p.max():.0%})")

'estou com uma forte dor no peito e muita falta de ar'
   -> alto risco (confiança 65%)
'estou apenas um pouco cansado depois de caminhar'
   -> baixo risco (confiança 64%)
'senti uma leve tontura mas já estou melhor'
   -> baixo risco (confiança 63%)
'meu coração está muito acelerado e estou me sentindo fraco'
   -> alto risco (confiança 63%)
'meu braço esquerdo está dormente e sinto náusea'
   -> alto risco (confiança 60%)
'tive uma leve dor no peito e falta de ar'
   -> alto risco (confiança 56%)
'não sinto dor no peito'
   -> alto risco (confiança 64%)
'estou com muita dor de cabeça'
   -> alto risco (confiança 55%)


## 8. Conclusão

**Resultado:** o modelo de Regressão Logística acertou 100% das 12 frases de teste (6 de cada classe), e a Árvore de Decisão ficou em 75%. Porém, com tão poucos dados, uma única divisão engana: na **validação cruzada** a Regressão Logística teve em média cerca de **92,5%** de acurácia (variando entre 88% e 100%). Esse é um número mais honesto para descrever o desempenho.

**O que o modelo aprendeu:** ele se apoia em palavras de *intensidade/gravidade*. Termos como "leve", "pouco" e "depois" empurram para **baixo risco**, enquanto "dor no peito", "suor", "respirar" e "palpitações" empurram para **alto risco**. Isso é sensato, mas mostra que ele "lê" vocabulário, e não o significado clínico.

**Padrões e distorções observados nos testes com frases novas:**
- **Negação não é entendida:** "não sinto dor no peito" foi classificada como alto risco, porque as palavras "dor" e "peito" pesam mais do que o "não".
- **Palavras fora do contexto cardíaco** também disparam alerta: "muita dor de cabeça" foi classificada como alto risco por causa da palavra "dor".
- **Mistura de gravidade e sintoma:** "leve dor no peito e falta de ar" vai para alto risco com baixa confiança (~56%), pois sinais de gravidade e de leveza competem. Nesses casos de confiança baixa, um sistema real deveria encaminhar para revisão humana.
- Palavras sem valor clínico ("está", "um", "depois") aparecem entre as mais influentes, sinal de que a base pequena gera **correlações acidentais**.
- A base é **simulada**, escrita por poucas pessoas e com um estilo de linguagem parecido, então o modelo não representa a diversidade de como pacientes reais se expressam (regionalismos, gírias, escolaridade), o que pode gerar **viés** contra quem fala de forma diferente.

**Em uma aplicação real**, seriam necessários: uma base muito maior, diversa e validada por profissionais de saúde; tratamento de negação; métricas que priorizem o **recall do alto risco** (deixar passar um caso grave é pior do que um falso alarme); e supervisão humana sempre. O sistema aqui é **exclusivamente acadêmico** e demonstra de forma simplificada como NLP e Machine Learning podem apoiar a triagem de relatos médicos.